In [ ]:
import torch
from unsloth import FastLanguageModel
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from pathlib import Path

persist_directory = "/workspaces/Arch_PC_Assistent/embed_model"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vectorstore = Chroma(embedding_function=embeddings,
                     persist_directory=persist_directory)


path_model_v1 = "/workspaces/Arch_PC_Assistent/outputs/SFT/arch_assistant_lora_1"
max_seq_length = 4096 
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = path_model_v1,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

In [ ]:
test_frage = "My Waybar is crashing on startup. How can I debug or fix it?"

docs = vectorstore.similarity_search(test_frage, k=5)
rag_context = "\n\n".join([f"--- Chunk {i+1} ({d.metadata.get('topic')}) ---\n{d.page_content}" for i, d in enumerate(docs)])

system_prompt = f"""You are ArchAgent, an expert Arch Linux assistant. 
Use ONLY the following context to answer the user's question. If the answer is not in the context, say so.
Your output MUST follow this format:
<think>
[Your reasoning here]
</think>
<answer>
[Your final concise answer here]
</answer>
CONTEXT:
{rag_context}"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_frage}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")


outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    use_cache=True,
    do_sample=True,      # ensure the variance of text generation 
    temperature=0.7,
    top_p=0.9,
    num_return_sequences=6
)

# Ergebnisse anzeigen
decoded_outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

for i, answer in enumerate(decoded_outputs):
    print(f"OUTPUT {i+1} ")
    print(answer)
    print("\n")

In [ ]:
import json
from tqdm import tqdm


input_file = "/workspaces/Arch_PC_Assistent/dataset/master_fragen_liste.json" 
output_file = "/workspaces/Arch_PC_Assistent/dataset/raw_generations.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    question_list = json.load(f)

print(f"Starting generation pipeline for {len(question_list)} questions...")

# Open in "a" (append) mode to prevent overwriting existing data
with open(output_file, "a", encoding="utf-8") as outfile:
    
    for question in tqdm(question_list, desc="Generating raw data"):
        
        # --- RAG Search (k=4 for VRAM protection) ---
        docs = vectorstore.similarity_search(question, k=4) 
        rag_context = "\n\n".join([f"--- Chunk {i+1} ({d.metadata.get('topic', 'unknown')}) ---\n{d.page_content}" for i, d in enumerate(docs)])

        # Improved system prompt with formatting rules at the very bottom
        system_prompt = f"""You are ArchAgent, an expert Arch Linux with hyprland and zsh setup assistant. 
        Below is retrieved context from the Arch Wiki. Use this context as your primary source of truth to ensure accuracy.

        CONTEXT:
        {rag_context}

        CRITICAL INSTRUCTIONS:
        If the context is incomplete or does not fully cover the user's problem, you MUST seamlessly use your own internal knowledge to provide a complete solution.

        Your output MUST strictly follow this exact format:
        <think>
        [Analyze the user's problem. Check what information is available in the context. If something is missing, retrieve it from your internal knowledge. Plan your solution.]
        </think>
        <answer>
        [Your final, clear, and actionable answer here.]
        </answer>"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
        
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        # Security limit of 2048 to ensure the question is not cut off
        inputs = tokenizer([prompt], return_tensors="pt", truncation=True, max_length=2048).to("cuda")

        # --- Generation (6 samples) ---
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            use_cache=True,
            do_sample=True,      
            temperature=0.7,
            top_p=0.9,
            num_return_sequences=6
        )

        # --- Decoding and Saving ---
        decoded_outputs = tokenizer.batch_decode(outputs[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        
        raw_sample = {
            "instruction": question,
            "input": rag_context,
            "generated_outputs": decoded_outputs # List with 6 text samples
        }
        
        outfile.write(json.dumps(raw_sample, ensure_ascii=False) + "\n")
        outfile.flush() # Force write data to disk immediately

print("\nGeneration completed! Your raw data is available in 'raw_generations.jsonl'.")